In [ ]:
!pip install praat-parselmouth

In [ ]:
from google.colab import drive
import pandas as pd
import ipywidgets as widgets
import librosa
import librosa.display
import os
# Import praat-parselmouth to use Praat's pitch tracking algorithm
# Needs installing once via !pip install praat-parselmouth (using the above cell)
import parselmouth
import matplotlib.pyplot as plt
import numpy as np
# UI that allows filtering, displaying sounds nicely
from IPython.display import display, clear_output, Audio

# 1. Mount Drive (Force remount to see the new shortcut)
drive.mount('/content/drive', force_remount=True)

# 2. Define the Path
PROJECT_PATH = '/content/drive/MyDrive/Zhuang'
METADATA_PATH = os.path.join(PROJECT_PATH, 'FileInfoForRIPA.csv')

# 3. specific check
if os.path.exists(PROJECT_PATH):
    print(f"✅ Found the Zhuang folder at: {PROJECT_PATH}")

    if os.path.exists(METADATA_PATH):
        df = pd.read_csv(METADATA_PATH)
        print("✅ Metadata loaded!")
        display(df.head())
    else:
        print(f"❌ Folder found, but could not find 'FileInfoForRIPA.csv' inside it.")
        print("Check if the file is still uploading?")
else:
    print(f"❌ Still cannot find the folder at: {PROJECT_PATH}")
    print("Did you create the shortcut in 'My Drive'?")

# ***********************************************

AUDIO_PATH = os.path.join(PROJECT_PATH, 'Audio')

# Create UI Widgets
tone_dropdown = widgets.Dropdown(
    options=['All'] + sorted(df['ReclassifiedTone'].dropna().unique().tolist()),
    value='All',
    description='Tone:'
)

vowel_length_dropdown = widgets.Dropdown(
    options=['All'] + sorted(df['ReclassifiedVowelLength'].dropna().unique().tolist()),
    value='All',
    description='Vowel Length:'
)

output_box = widgets.Output()

# Plotting function
def plot_audio_files(tone, vowel_length, n):
  with output_box:
    clear_output(wait=True)

    # Filter based on selections
    filtered_df = df.copy()
    if tone != 'All':
      filtered_df = filtered_df[filtered_df['ReclassifiedTone'] == tone]
    if vowel_length != 'All':
      filtered_df = filtered_df[filtered_df['ReclassifiedVowelLength'] == vowel_length]

    print(f"Found {len(filtered_df)} matching files. Displaying the first {n}...")

    # Limit to the first n files
    for index, row in filtered_df.head(n).iterrows():
      # Construct the file path
      file_name = row['Filename'] + '.wav'
      audio_path = os.path.join(AUDIO_PATH, file_name)

      if not os.path.exists(audio_path):
        print(f"Missing audio file: {file_name}")
        continue

      print(f"\n--- File: {file_name} | Tone: {row['ReclassifiedTone']} | Vowel Length: {row['ReclassifiedVowelLength']}")

      # Load audio
      y, sr = librosa.load(audio_path, sr = None)

      # Load with parselmouth for f0
      snd = parselmouth.Sound(audio_path)
      pitch = snd.to_pitch()
      pitch_values = pitch.selected_array['frequency']
      pitch_times = pitch.xs()

      # Generate spectrogram
      D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)

      # Create 3-panel figure
      fig, ax = plt.subplots(nrows=3, ncols = 1, figsize = (10,8), sharex=True)

      # Panel 1 - waveform
      librosa.display.waveshow(y, sr = sr, ax = ax[0])
      ax[0].set(title = 'Waveform', ylabel = 'Amplitude')

      # Panel 2 - spectrogram
      librosa.display.specshow(D, sr = sr, x_axis = 'time', y_axis = 'hz', ax = ax[1])
      ax[1].set(title = 'Spectrogram', ylabel = 'Hz')

      # Panel - pitch track (I'll try to use an overlay in future though)
      pitch_values[pitch_values == 0] = np.nan # Hide voiceless segments
      ax[2].plot(pitch_times, pitch_values, 'o', markersize = 2)
      ax[2].set_facecolor('black') # background color
      ax[2].set(title = 'Pitch (f0)', ylabel = 'Frequency (Hz)', xlabel = 'Time (s)')
      ax[2].set_ylim(50, 500) # pitch range

      plt.tight_layout()
      plt.show()

      # Add audio player
      display(Audio(data = y, rate = sr))

# Link widgets to function
widgets.interactive(plot_audio_files, tone = tone_dropdown, vowel_length = vowel_length_dropdown, n = 10)

# Display the UI
display(widgets.HBox([tone_dropdown, vowel_length_dropdown]))
display(output_box)